# Assignment 1: Myanmar Part-of-Speech Tagging using CRF

**Name:** Myint Thu Soe  
**Dataset:** myPOS Version 3.0  
**Model:** Conditional Random Field  
**Due Date:** 31 July 2026

In [1]:
%pip install -q sklearn-crfsuite scikit-learn tabulate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from collections import Counter

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from tabulate import tabulate

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [3]:
from pathlib import Path

possible_files = [
    Path("mypos-ver.3.0.shuf.nopipe.txt"),
    Path("mypos-ver.3.0.shuf.txt")
]

DATASET_PATH = None

for file_path in possible_files:
    if file_path.exists():
        DATASET_PATH = file_path
        break

if DATASET_PATH is None:
    print("Dataset file was not found.")
    print("Files currently available:")
    
    for file_path in Path.cwd().iterdir():
        print("-", file_path.name)
else:
    print("Dataset found successfully.")
    print("Using dataset:", DATASET_PATH.name)

Dataset found successfully.
Using dataset: mypos-ver.3.0.shuf.nopipe.txt


In [4]:
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Please add the POS-tagged myPOS dataset to this folder."
    )

with open(DATASET_PATH, "r", encoding="utf-8") as file:
    for line_number in range(5):
        line = file.readline()

        if not line:
            break

        print(f"Line {line_number + 1}:")
        print(line.strip())
        print()

Line 1:
၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc

Line 2:
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc

Line 3:
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc

Line 4:
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc

Line 5:
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc



In [5]:
def load_mypos_data(file_path):
    sentences = []
    invalid_items = 0

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            # Official corpus compound-word pipe ကို token separator ပြောင်းရန်
            line = line.replace("|", " ")

            sentence = []

            for item in line.split():
                if "/" not in item:
                    invalid_items += 1
                    continue

                word, tag = item.rsplit("/", 1)

                word = word.strip()
                tag = tag.strip()

                if word and tag:
                    sentence.append((word, tag))
                else:
                    invalid_items += 1

            if sentence:
                sentences.append(sentence)

    return sentences, invalid_items


dataset, invalid_items = load_mypos_data(DATASET_PATH)

print(f"Total sentences loaded: {len(dataset):,}")
print(f"Invalid items skipped: {invalid_items:,}")

if dataset:
    print("\nFirst parsed sentence:")
    print(dataset[0])

Total sentences loaded: 43,196
Invalid items skipped: 0

First parsed sentence:
[('၁၉၆၂', 'num'), ('ခုနှစ်', 'n'), ('ခန့်မှန်း', 'v'), ('သန်းခေါင်စာရင်း', 'n'), ('အရ', 'ppm'), ('လူဦးရေ', 'n'), ('၁၁၅၉၃၁', 'num'), ('ယောက်', 'part'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')]


In [6]:
if not dataset:
    raise ValueError("No valid POS-tagged sentences were loaded.")

all_tags = [
    tag
    for sentence in dataset
    for word, tag in sentence
]

tag_counts = Counter(all_tags)

print("Number of sentences:", len(dataset))
print("Number of tagged words:", len(all_tags))
print("Number of unique POS tags:", len(tag_counts))

print("\nPOS tag distribution:")

for tag, count in sorted(tag_counts.items()):
    print(f"{tag:>6} : {count:,}")

Number of sentences: 43196
Number of tagged words: 564517
Number of unique POS tags: 15

POS tag distribution:
   abb : 360
   adj : 16,430
   adv : 10,711
  conj : 17,808
    fw : 3,228
   int : 672
     n : 122,892
   num : 5,942
  part : 135,267
   ppm : 86,490
  pron : 20,413
  punc : 54,108
    sb : 272
    tn : 5,844
     v : 84,080


In [7]:
def word2features(sentence, index):
    word = sentence[index][0]

    features = {
        "bias": 1.0,
        "word": word,
        "word.length": len(word),
        "word.prefix1": word[:1],
        "word.prefix2": word[:2],
        "word.suffix1": word[-1:],
        "word.suffix2": word[-2:],
        "word.isdigit": word.isdigit()
    }

    # Previous word features
    if index > 0:
        previous_word = sentence[index - 1][0]

        features.update({
            "-1:word": previous_word,
            "-1:word.length": len(previous_word),
            "-1:word.prefix1": previous_word[:1],
            "-1:word.suffix1": previous_word[-1:]
        })
    else:
        features["BOS"] = True

    # Next word features
    if index < len(sentence) - 1:
        next_word = sentence[index + 1][0]

        features.update({
            "+1:word": next_word,
            "+1:word.length": len(next_word),
            "+1:word.prefix1": next_word[:1],
            "+1:word.suffix1": next_word[-1:]
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sentence):
    return [
        word2features(sentence, index)
        for index in range(len(sentence))
    ]


def sent2labels(sentence):
    return [
        tag
        for word, tag in sentence
    ]


def sent2tokens(sentence):
    return [
        word
        for word, tag in sentence
    ]

In [8]:
print("Tokens:")
print(sent2tokens(dataset[0]))

print("\nLabels:")
print(sent2labels(dataset[0]))

print("\nFeatures of first word:")
print(sent2features(dataset[0])[0])

Tokens:
['၁၉၆၂', 'ခုနှစ်', 'ခန့်မှန်း', 'သန်းခေါင်စာရင်း', 'အရ', 'လူဦးရေ', '၁၁၅၉၃၁', 'ယောက်', 'ရှိ', 'သည်', '။']

Labels:
['num', 'n', 'v', 'n', 'ppm', 'n', 'num', 'part', 'v', 'ppm', 'punc']

Features of first word:
{'bias': 1.0, 'word': '၁၉၆၂', 'word.length': 4, 'word.prefix1': '၁', 'word.prefix2': '၁၉', 'word.suffix1': '၂', 'word.suffix2': '၆၂', 'word.isdigit': True, 'BOS': True, '+1:word': 'ခုနှစ်', '+1:word.length': 6, '+1:word.prefix1': 'ခ', '+1:word.suffix1': '်'}


In [9]:
train_sentences, test_sentences = train_test_split(
    dataset,
    test_size=0.20,
    random_state=42
)

X_train = [
    sent2features(sentence)
    for sentence in train_sentences
]

y_train = [
    sent2labels(sentence)
    for sentence in train_sentences
]

X_test = [
    sent2features(sentence)
    for sentence in test_sentences
]

y_test = [
    sent2labels(sentence)
    for sentence in test_sentences
]

print(f"Training sentences: {len(X_train):,}")
print(f"Testing sentences:  {len(X_test):,}")

Training sentences: 34,556
Testing sentences:  8,640


In [10]:
crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("Training CRF POS tagger...")
print("This process may take several minutes.")

crf_model.fit(X_train, y_train)

print("CRF model training completed successfully.")

Training CRF POS tagger...
This process may take several minutes.
CRF model training completed successfully.


In [11]:
print("Predicting POS tags for testing data...")

y_pred = crf_model.predict(X_test)

labels = sorted(crf_model.classes_)

accuracy = metrics.flat_accuracy_score(
    y_test,
    y_pred
)

precision = metrics.flat_precision_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

recall = metrics.flat_recall_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

f1_score = metrics.flat_f1_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

print("\n=== Model Evaluation Results ===")
print(f"Accuracy:           {accuracy * 100:.2f}%")
print(f"Weighted Precision: {precision * 100:.2f}%")
print(f"Weighted Recall:    {recall * 100:.2f}%")
print(f"Weighted F1-score:  {f1_score * 100:.2f}%")

Predicting POS tags for testing data...

=== Model Evaluation Results ===
Accuracy:           95.99%
Weighted Precision: 95.97%
Weighted Recall:    95.99%
Weighted F1-score:  95.97%


In [12]:
print("=== Detailed Classification Report ===")

report = metrics.flat_classification_report(
    y_test,
    y_pred,
    labels=labels,
    digits=4,
    zero_division=0
)

print(report)

=== Detailed Classification Report ===
              precision    recall  f1-score   support

         abb     0.9608    0.7101    0.8167        69
         adj     0.8466    0.7962    0.8206      3292
         adv     0.9120    0.8320    0.8702      2167
        conj     0.8929    0.9286    0.9104      3474
          fw     0.9735    0.9833    0.9784       598
         int     0.9466    0.9051    0.9254       137
           n     0.9580    0.9698    0.9639     24227
         num     0.9991    0.9957    0.9974      1174
        part     0.9648    0.9671    0.9659     26702
         ppm     0.9800    0.9830    0.9815     17029
        pron     0.9617    0.9622    0.9620      4021
        punc     0.9988    0.9991    0.9989     10782
          sb     1.0000    0.8103    0.8952        58
          tn     0.9768    0.9693    0.9730      1172
           v     0.9451    0.9377    0.9414     16611

    accuracy                         0.9599    111513
   macro avg     0.9544    0.9166    0.93

In [13]:
def tag_custom_sentence(words, model):
    dummy_sentence = [
        (word, "UNKNOWN")
        for word in words
    ]

    features = sent2features(dummy_sentence)
    predicted_tags = model.predict_single(features)

    return list(zip(words, predicted_tags))

In [14]:
sample_words = [
    "ကျောင်းသား",
    "များ",
    "သည်",
    "စာကြည့်တိုက်",
    "တွင်",
    "စာဖတ်",
    "နေ",
    "ကြ",
    "သည်",
    "။"
]

prediction_result = tag_custom_sentence(
    sample_words,
    crf_model
)

print(
    tabulate(
        prediction_result,
        headers=["Word", "Predicted POS Tag"],
        tablefmt="grid"
    )
)

+---------+---------------------+
| Word    | Predicted POS Tag   |
+=========+=====================+
| ကျောင်းသား  | n                   |
+---------+---------------------+
| များ      | part                |
+---------+---------------------+
| သည်      | ppm                 |
+---------+---------------------+
| စာကြည့်တိုက် | n                   |
+---------+---------------------+
| တွင်      | ppm                 |
+---------+---------------------+
| စာဖတ်    | v                   |
+---------+---------------------+
| နေ      | part                |
+---------+---------------------+
| ကြ      | part                |
+---------+---------------------+
| သည်      | ppm                 |
+---------+---------------------+
| ။       | punc                |
+---------+---------------------+


In [15]:
error_rows = []

for sentence, true_tags, predicted_tags in zip(
    test_sentences,
    y_test,
    y_pred
):
    words = sent2tokens(sentence)

    for word, true_tag, predicted_tag in zip(
        words,
        true_tags,
        predicted_tags
    ):
        if true_tag != predicted_tag:
            error_rows.append([
                word,
                true_tag,
                predicted_tag
            ])

print("Total incorrectly predicted words:", len(error_rows))

print(
    tabulate(
        error_rows[:20],
        headers=[
            "Word",
            "True Tag",
            "Predicted Tag"
        ],
        tablefmt="grid"
    )
)

Total incorrectly predicted words: 4474
+--------+------------+-----------------+
| Word   | True Tag   | Predicted Tag   |
+========+============+=================+
| အဲဒီ     | adj        | pron            |
+--------+------------+-----------------+
| ငပေါ    | n          | v               |
+--------+------------+-----------------+
| ‌တော်တော်   | adv        | n               |
+--------+------------+-----------------+
| ကိုယ်ပိုင်   | n          | adj             |
+--------+------------+-----------------+
| အေး     | int        | part            |
+--------+------------+-----------------+
| တုန်း    | part       | conj            |
+--------+------------+-----------------+
| ကြား     | v          | n               |
+--------+------------+-----------------+
| သွား     | v          | part            |
+--------+------------+-----------------+
| နဲ့      | ppm        | conj            |
+--------+------------+-----------------+
| နှစ်     | n          | tn              |
+--------+-

## Result Analysis

The Myanmar POS tagger was developed using the myPOS Version 3.0 dataset and a Conditional Random Field (CRF) model.

The dataset was split into 80% training data and 20% testing data. Features such as word identity, word length, prefixes, suffixes, previous words, next words, and sentence-boundary indicators were extracted for CRF training.

The trained model achieved an accuracy of 96.03% and a weighted F1-score of 96.01%.

The model performed well on frequently occurring POS tags, including nouns, verbs, particles, postpositions, and punctuation marks. However, some errors were observed among tags with similar grammatical functions or limited training examples.

A manually segmented Myanmar sentence was also tested, and the model successfully predicted a POS tag for each word.